# Агрегатные признаки по датчикам комнат

В этом ноутбуке создаются общие показатели температуры и влажности по датчикам внутри дома. Они помогают модели видеть состояние дома целиком, а не анализировать каждый датчик отдельно.

In [1]:
from pathlib import Path

import pandas as pd


## Загрузка данных

Ноутбук можно запускать как из папки `notebooks/`, так и из корня репозитория.

In [2]:
data_paths = [
    Path('../data/energydata_complete.csv'),
    Path('data/energydata_complete.csv'),
]
data_path = next((path for path in data_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError('Не найден data/energydata_complete.csv')

df = pd.read_csv(data_path)
df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


## Списки датчиков

Используются внутренние датчики `T1`-`T5`, `T7`-`T9` и соответствующие им датчики влажности. `T6` и `RH_6` не включаются: по описанию датасета это внешние датчики на северной стороне здания.

In [3]:
indoor_sensor_numbers = [1, 2, 3, 4, 5, 7, 8, 9]
temperature_columns = [f'T{i}' for i in indoor_sensor_numbers]
humidity_columns = [f'RH_{i}' for i in indoor_sensor_numbers]

missing_columns = set(temperature_columns + humidity_columns) - set(df.columns)
if missing_columns:
    raise KeyError(f'В датасете нет столбцов: {sorted(missing_columns)}')


## Создание агрегатных признаков

Для температуры и влажности рассчитываются среднее, минимум, максимум и разброс между самым большим и самым маленьким значением. Также добавляются разницы между средней температурой/влажностью внутри дома и на улице.

In [4]:
df['indoor_temperature_mean'] = df[temperature_columns].mean(axis=1)
df['indoor_temperature_min'] = df[temperature_columns].min(axis=1)
df['indoor_temperature_max'] = df[temperature_columns].max(axis=1)
df['indoor_temperature_range'] = (
    df['indoor_temperature_max'] - df['indoor_temperature_min']
)

df['indoor_humidity_mean'] = df[humidity_columns].mean(axis=1)
df['indoor_humidity_min'] = df[humidity_columns].min(axis=1)
df['indoor_humidity_max'] = df[humidity_columns].max(axis=1)
df['indoor_humidity_range'] = (
    df['indoor_humidity_max'] - df['indoor_humidity_min']
)

df['indoor_outdoor_temperature_diff'] = df['indoor_temperature_mean'] - df['T_out']
df['indoor_outdoor_humidity_diff'] = df['indoor_humidity_mean'] - df['RH_out']

aggregate_features = [
    'indoor_temperature_mean',
    'indoor_temperature_min',
    'indoor_temperature_max',
    'indoor_temperature_range',
    'indoor_humidity_mean',
    'indoor_humidity_min',
    'indoor_humidity_max',
    'indoor_humidity_range',
    'indoor_outdoor_temperature_diff',
    'indoor_outdoor_humidity_diff',
]

df[aggregate_features].head()

,indoor_temperature_mean,indoor_temperature_min,indoor_temperature_max,indoor_temperature_range,indoor_humidity_mean,indoor_humidity_min,indoor_humidity_max,indoor_humidity_range,indoor_outdoor_temperature_diff,indoor_outdoor_humidity_diff
0,18.435000,17.033333,19.89,2.856667,46.742500,41.626667,55.20,13.573333,11.835000,-45.257500
1,18.439167,17.066667,19.89,2.823333,46.672708,41.560000,55.20,13.640000,11.955833,-45.327292
2,18.421667,17.000000,19.89,2.890000,46.562917,41.433333,55.09,13.656667,12.055000,-45.437083
3,18.396250,17.000000,19.89,2.890000,46.468750,41.290000,55.09,13.800000,12.146250,-45.531250
4,18.408750,17.000000,19.89,2.890000,46.462917,41.230000,55.09,13.860000,12.275417,-45.537083


## Проверка результата

Проверяем, что новые признаки созданы для всех строк, не содержат пропусков и что разброс не бывает отрицательным.

In [5]:
assert df[aggregate_features].isna().sum().sum() == 0, 'В новых признаках есть пропуски'
assert (df['indoor_temperature_range'] >= 0).all(), 'Некорректный разброс температуры'
assert (df['indoor_humidity_range'] >= 0).all(), 'Некорректный разброс влажности'

df[aggregate_features].describe().T

,count,mean,std,min,25%,50%,75%,max
indoor_temperature_mean,19735.0,20.815611,1.812567,16.012708,19.663000,20.597500,21.764375,26.061940
indoor_temperature_min,19735.0,18.993190,1.866267,14.890000,17.760000,18.600000,20.100000,24.500000
indoor_temperature_max,19735.0,22.779580,1.965423,17.290000,21.587302,22.600000,23.963333,29.856667
indoor_temperature_range,19735.0,3.786390,0.907657,1.260000,3.132048,3.700000,4.320278,7.770000
indoor_humidity_mean,19735.0,41.221952,3.977196,30.842798,38.354583,40.572708,44.162176,53.947153
indoor_humidity_min,19735.0,35.067213,4.863485,20.463333,31.390000,34.590000,38.565833,47.218000
indoor_humidity_max,19735.0,51.143511,8.920412,35.363333,45.560000,49.320000,53.900000,96.321667
indoor_humidity_range,19735.0,16.076298,8.559847,4.984286,11.000000,13.718000,17.642778,63.821667
indoor_outdoor_temperature_diff,19735.0,13.403946,4.202446,-1.280083,10.668958,13.589667,16.293631,24.172708
indoor_outdoor_humidity_diff,19735.0,-38.528466,13.662339,-61.735631,-49.001875,-41.474653,-29.673345,9.981333


### Итог

Добавлены десять агрегатных признаков. На этапе моделирования они могут использоваться вместе с исходными показаниями датчиков, внешними метеоданными и временными признаками.